### Tutorial 5 — Cell phenotyping on cHL with a KRONOS2 linear probe

Tutorial 3 gave every cell a **feature vector** and a **ground-truth cell type**. This tutorial asks the question those two ingredients exist to answer: *how much cell-type information is actually in a KRONOS2 embedding?*

The standard way to measure that is a **linear probe** — freeze the encoder, fit a plain multinomial logistic regression on top, and report how well it separates the classes. Nothing is fine-tuned, so the score is a property of the **representation**, not of the classifier.

1. **Load** per-cell features + labels from Tutorial 3, joined by `cell_id`
2. **Spatial folds** — split the slide into four quadrants with a guard band, so train and test are never spatial neighbours
3. **Linear probe** — standardize, then tune the regularization strength `C` with Optuna on a held-out validation split
4. **Evaluate** — macro F1, balanced accuracy, average precision, ROC AUC, per fold and averaged
5. **Baseline** — the same probe on mean-marker features, so the KRONOS2 number has something to beat
6. **Visualize** — confusion matrix and a spatial map of the out-of-fold predictions

Every protocol choice below — which classes are scored, how many cells the probe trains on, the `C` search space, the trial count — follows the published KRONOS cell-phenotyping benchmark, so the numbers land on the same footing as the ones in the paper. Section 9 is candid about the one thing that still differs.

> **Prerequisites.** Run [Tutorial 0](0-Example-Data-Download.ipynb) (data), [Tutorial 1](1-Tissue-Ingest.ipynb) (ingest), [Tutorial 2](2-Step-by-Step-Patch-Feature-Extraction.ipynb) (tissue), and [Tutorial 3](3-Cell-Segmentation-and-Feature-Extraction.ipynb) (cell mask, labels, **and both feature extractions**) first. This tutorial does **no** segmentation, patching, or feature extraction — it reads what Tutorial 3 already wrote into the slide store. Launch Jupyter from either `tutorials/` or the repo root.

Unlike Tutorials 1–4 there are **no CLI commands** here. Fold design, probing, and evaluation are downstream of CORAL — CORAL's job ends at the feature matrix, and what you do with it is ordinary Python.

#### 0 — Installation

Two dependencies that are **not** CORAL dependencies — CORAL deliberately stops at features and leaves modelling to the consumer — so install them into the tutorial environment directly:

```bash
uv pip install scikit-learn optuna
```

scikit-learn provides the probe; Optuna tunes its one hyperparameter. No GPU needed for this notebook. The one step that wanted a GPU (KRONOS2 feature extraction) already happened in Tutorial 3; here we only read the stored vectors.

#### 1. Load the per-cell features and labels

Reopen the slide store and rebuild the exact `PatchConfig` Tutorial 3 used — the config resolves to a slug (`cell_0.37mpp_64px`) that names the patch set on disk, so it has to match.

Passing the extractor **by name** (the string `"KRONOS2"`) reads the stored array directly: no weights are loaded, no GPU is touched.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from coral import CoralSlide
from coral.config import PatchConfig
from coral.config.subset import Selection

# Locate Tutorial 1's output, whether Jupyter launched from tutorials/ or root.
cwd = Path.cwd()
tutorials_dir = cwd if (cwd / "utils").exists() else cwd / "tutorials"
OUTPUT_DIR = tutorials_dir / "example-data" / "processed"

CELL_PATCH_SIZE = 64   # must match Tutorial 3
SEED = 42              # pins every sampling step (folds, validation, capping)

# The 18 phenotypic markers Tutorial 3 extracted on (and Tutorial 4 clusters on).
PANEL = [
    "dapi", "cd11b", "cd11c", "cd15", "cd163", "cd20", "cd206", "cd30",
    "cd31", "cd4", "cd56", "cd68", "cd7", "cd8", "cytokeratin", "foxp3",
    "mct", "podoplanin",
]

slide = CoralSlide.open(OUTPUT_DIR / "raw_image.zarr")
cell_cfg = PatchConfig(patch_size=CELL_PATCH_SIZE, mode="cell_centered")
panel = Selection(include=PANEL)
print(slide)
print("cell patch set:", cell_cfg.name, "| panel:", len(PANEL), "markers")

Tutorial 3 extracted with `--subset panel.yaml`, so the stored variant is `markers_panel` (named after the subset file) — read it back by that variant name. A mismatched variant name raises rather than silently reading a different array; if you extracted over all channels instead, the variant is `markers_all`.

In [ ]:
def load_cell_features(slide, extractor_name, config, *, variant="markers_panel"):
    """Read a stored per-cell feature set, or explain what's missing."""
    try:
        return slide.features(extractor_name, config, suffix=variant)
    except (FileNotFoundError, ValueError) as err:
        raise RuntimeError(
            f"No stored '{extractor_name}' features for patch set "
            f"'{config.name}' in {slide.path}.\n"
            "Run Tutorial 3 (Cell Segmentation & Feature Extraction) first — "
            "its step 4 writes both the mean-marker and the KRONOS2 per-cell "
            "features this notebook reads, on the markers_panel variant."
        ) from err


emb = load_cell_features(slide, "KRONOS2", cell_cfg)
print("KRONOS2 per-cell features:", emb["features"].shape, "(n_cells, 768)")
emb

##### Joining features to ground truth

A cell-centered feature set carries a **`cell_id`** coordinate (grid patch sets carry `x`/`y` instead). Labels join on that id. Two things to handle:

- **Not every cell is labelled.** The mask has ~153k cells; the annotation file covers ~145k. Unlabelled cells drop out.
- **`cell_labels.csv` can hold several label sets** stacked in one file, distinguished by `label_set` — filter to the one you want (`"maps"`, imported in Tutorial 3).

Centroids come from `cell_centroids.csv`, not from the features: for cell-centered patches the stored coords are the patch **top-left**, whereas the fold split below needs the cell's actual centre.

In [ ]:
cell_ids = emb.coords["cell_id"].values

labels_df = pd.read_csv(slide.path / "cells" / "cell_labels.csv")
labels_df = labels_df[labels_df["label_set"] == "maps"].set_index("cell_id")

centroids = pd.read_csv(slide.path / "cells" / "cell_centroids.csv").set_index("cell_id")

cells = pd.DataFrame(
    {
        "label": labels_df["label"].reindex(cell_ids).values,
        "x": centroids["x"].reindex(cell_ids).values,
        "y": centroids["y"].reindex(cell_ids).values,
    },
    index=pd.Index(cell_ids, name="cell_id"),
)

labelled = cells["label"].notna().values
print(f"{labelled.sum():,} labelled of {len(cells):,} cells with features")

#### 2. The label set

The raw annotation is not quite a list of cell types, and turning it into one takes two decisions.

- **`Seg Artifact` is dropped.** These are segmentation failures — cells that are not cells. Scoring a phenotyping model on them measures the segmenter, not the encoder.
- **`Cytotoxic CD8` is merged into `CD8`.** It is a genuine subtype, but with ~380 cells against CD8's ~17k it is too rare to score reliably, and holding it out separately would *also* remove those cells from CD8. Merging keeps both classes honest.

`Other` is **kept** as a class. It is tempting to drop — it is a catch-all with no single consistent phenotype, so it is hard in a way that feels unfair. That is exactly why it stays: it is populous, it is the class a real pipeline most needs to get right, and a benchmark that quietly discards its hardest class reports a number that cannot be compared with one that doesn't.

That leaves **16 cell types**. 

Note how skewed the classes are — CD4 T cells outnumber the rarest types by more than 10:1. That imbalance is why the probe uses `class_weight="balanced"` and why **macro** F1 (which weights every class equally) is the headline metric rather than accuracy.

In [ ]:
# The published benchmark's class list, in its label-ID order.
BENCHMARK_CLASS_NAMES = [
    "B", "CD4", "CD8", "DC", "Endothelial", "Epithelial", "Lymphatic",
    "M1", "M2", "Mast", "Monocyte", "NK", "Neutrophil", "Other",
    "TReg", "Tumor",
]

keep = labelled & (cells["label"] != "Seg Artifact").values
cells = cells[keep]
X_kronos = np.asarray(emb["features"].values, dtype=np.float32)[keep]

# Merge before the integer codes are assigned, so CD8 absorbs the subtype.
n_merged = int((cells["label"] == "Cytotoxic CD8").sum())
cells["label"] = cells["label"].replace({"Cytotoxic CD8": "CD8"})
print(f"merged {n_merged:,} 'Cytotoxic CD8' cells into 'CD8'")

CLASS_NAMES = sorted(cells["label"].unique())
assert CLASS_NAMES == BENCHMARK_CLASS_NAMES, (
    f"class list drifted from the benchmark:\n"
    f"  here:      {CLASS_NAMES}\n"
    f"  benchmark: {BENCHMARK_CLASS_NAMES}"
)
LABELS = np.arange(len(CLASS_NAMES))
y = pd.Categorical(cells["label"], categories=CLASS_NAMES).codes.astype(np.int64)

print(f"{len(CLASS_NAMES)} classes, {len(cells):,} cells, X = {X_kronos.shape}")
cells["label"].value_counts()

#### 3. Spatial folds — why a random split would lie to you

The obvious thing is a random 80/20 split of the cells. On spatial data that **inflates the score**, for two reasons:

1. **Spatial autocorrelation.** Cells sitting side by side in the same germinal centre share microenvironment, staining, and illumination. Put one in train and its neighbour in test and the probe is being graded on something close to a memorised example.
2. **Patch overlap.** Cell-centered patches are 64 px boxes on centroids. Two cells 20 px apart produce patches sharing most of their pixels — the *same image content* on both sides of the split.

So we split **geographically**: quarter the slide, and hold out one quadrant at a time. Train and test are then physically separate tissue, which is the honest question — *does this generalize to tissue the probe has never seen?*

That still leaves leakage at the seams, where a cell just left of the midline has a patch reaching across it. A **guard band** of one patch width (64 px) either side of each midline fixes that: cells falling in the band are dropped from every fold rather than assigned to one.

This gives 4 folds. For each, one quadrant is **test**, the other three are **train**, and a random 20% of train is held out as **validation** — used only to pick `C`, never to score.

In [ ]:
_, height, width = slide.image.shape
band = CELL_PATCH_SIZE          # guard band half-width, in level-0 pixels
mid_x, mid_y = width // 2, height // 2

left = cells["x"] <= mid_x - band
right = cells["x"] >= mid_x + band
top = cells["y"] <= mid_y - band
bottom = cells["y"] >= mid_y + band

# Quadrant 1..4 (TL, TR, BL, BR); 0 = guard band, dropped.
quadrant = np.select(
    [left & top, right & top, left & bottom, right & bottom],
    [1, 2, 3, 4],
    default=0,
)

on_grid = quadrant > 0
print(f"dropped {(~on_grid).sum():,} cells in the guard band")

cells = cells[on_grid]
X_kronos = X_kronos[on_grid]
y = y[on_grid]
quadrant = quadrant[on_grid]

print(pd.Series(quadrant).value_counts().sort_index().rename("cells per quadrant"))

##### Look at the split before trusting it

Quadrants only work as folds if each one actually contains tissue and a reasonable spread of cell types. A quadrant that is 90% background — or missing a class entirely — produces a fold whose test metrics are meaningless. Plot it.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(15, 6))

ax[0].scatter(cells["x"], cells["y"], c=quadrant, cmap="Set2", s=0.4, alpha=0.6)
ax[0].axvline(mid_x, color="k", lw=0.8, ls="--")
ax[0].axhline(mid_y, color="k", lw=0.8, ls="--")
ax[0].set(title="Spatial folds (guard band removed)", xlabel="x", ylabel="y")
ax[0].invert_yaxis()
ax[0].set_aspect("equal")

composition = (
    pd.crosstab(cells["label"], quadrant, normalize="columns")
    .reindex(CLASS_NAMES)
)
im = ax[1].imshow(composition.values, cmap="viridis", aspect="auto")
ax[1].set(
    title="Class composition per quadrant",
    xticks=range(4), xticklabels=[f"Q{i}" for i in (1, 2, 3, 4)],
    yticks=range(len(CLASS_NAMES)), yticklabels=CLASS_NAMES,
)
fig.colorbar(im, ax=ax[1], label="fraction of quadrant")
plt.tight_layout()
plt.show()

The quadrants are not identically composed — one is tumour-rich, another lymphocyte-rich. That is a **feature** of this design: each fold tests generalization to a genuinely different tissue neighbourhood, and the spread across folds tells you how much the score depends on which region you held out. A tight standard deviation is evidence the representation is robust.

##### Drawing the splits

Two helpers, both built on `DataFrame.sample` with a pinned `random_state` so the whole notebook reruns identically:

- **`split_fold`** holds out one quadrant as test, concatenates the other three, and takes a random 20% of that as validation. The validation draw is *not* spatial — validation cells sit right next to training cells, so validation macro F1 is optimistic. That is fine for its only job, which is ranking candidate `C` values; it is never reported as a score.
- **`cap_per_class`** subsamples the training rows so no class contributes more than a fixed number. Why that matters is covered in section 4.

Everything downstream indexes by **row position** rather than `cell_id`, so the same indices address `X_kronos`, `y`, and `cells` interchangeably.

In [ ]:
# split_fold    — holds out one quadrant as the test set and splits the
#                 rest 80/20 into train/valid, so train and test cells are
#                 never spatial neighbours.
# cap_per_class — caps the training rows per class, so an abundant class
#                 cannot dominate the fit on volume alone.
from utils.cell_phenotyping_utils import cap_per_class, split_fold

for fold_id in (1, 2, 3, 4):
    tr, va, te = split_fold(quadrant, fold_id, random_state=SEED)
    print(f"fold {fold_id}: train {len(tr):>6,}  valid {len(va):>6,}  test {len(te):>6,}")


#### 4. The probe

Four conventions make a probe comparable across encoders. All of them matter more than the model choice:

- **Standardize on train statistics only.** `StandardScaler` is fit on the training rows and *applied* to validation and test. Fitting it on everything leaks test distribution into training.
- **`class_weight="balanced"`.** Under this much imbalance an unweighted probe learns to predict CD4 and stop. Balancing reweights the loss so rare classes carry equal total weight.
- **Cap cells per class.** Capping training at 2000 cells per class keeps the fit bounded *and* keeps the comparison honest — every encoder gets the same training budget, so a difference in score is a difference in representation, not in how much data each one happened to see. The cap only binds for the abundant classes; the rarest types have fewer than 2000 cells in total, so they contribute everything they have.
- **Search `C` on validation, score once on test.** `C` is the inverse regularization strength and it matters a lot: too small and the probe underfits to near-chance, too large and it overfits 768 dimensions to a few tens of thousands of cells.

##### Tuning `C` with Optuna

The search space is `C ∈ [1e-4, 1e2]`, sampled log-uniformly, with **15 trials** per fold and validation macro F1 as the objective.

Optuna's TPE sampler is adaptive: its first several draws are random, and after that it concentrates new trials where the observed scores were good. Fifteen decades is a wide space to cover in 15 draws, so that concentration is doing real work.

**On the sampler seed.** This notebook seeds the sampler by default, since a reproducible number is more useful in a tutorial; set `SEED_SAMPLER = False` to reproduce that non-determinism instead.

One consequence of seeding to expect below: a **fresh sampler is built per fold**, so every fold draws the *same* first ten `C` values — TPE samples randomly until `n_startup_trials` (default 10) have completed. Folds diverge only once TPE begins conditioning on observed scores, which differ per fold. Identical early trials across folds are therefore the seed working, not a bug. It does mean that cutting `N_TRIALS` below ~10 collapses the search into a fixed grid shared by every fold.

In [ ]:
# fit_probe — standardizes the features, tunes C with an Optuna TPE search
#             scored on validation macro F1, then refits the probe from
#             scratch on the winning C.
from utils.cell_phenotyping_utils import fit_probe

# The protocol this notebook claims parity with. Every value is passed
# explicitly into the helpers below, so the run is described here rather
# than buried in utils/.
N_TRIALS = 15                    # Optuna trials per fold
C_RANGE = (1e-4, 1e2)            # log-uniform search bounds on C
MAX_ITER = 10000
MAX_CELLS_PER_CLASS = 2000       # training budget per class; None = no cap
SEED_SAMPLER = True              # False = unseeded sampler, as published

SAMPLER_SEED = SEED if SEED_SAMPLER else None


##### Save the splits so other notebooks can reuse them

The fold assignment is the expensive part of the protocol to get right, and it is entirely independent of which encoder or model you probe. Write it out once, as plain `cell_id,label` CSVs, and any downstream experiment — a different classifier, an MLP head, a label-efficiency sweep — can load the *same* rows instead of re-deriving quadrants and hoping the seed matches.

Twelve files land in `example-data/cell-pheno-results/folds/`:

```
train_2000_fold1.csv   val_fold1.csv   test_fold1.csv
... one set per fold, 1..4
```

The train files carry the **capped** rows (`2000` in the name is `MAX_CELLS_PER_CLASS`), so the training budget is baked into the filename rather than left implicit. Validation and test are uncapped — they are scored as-is.

The cells written here are exactly what `run_cross_validation` uses below: same `split_fold` and `cap_per_class` calls, same `SEED`, so the CSVs and the probe never disagree.

In [ ]:
# save_split — writes one split's `cell_id,label` rows to CSV, so another
#              notebook can score a different model on exactly these cells.
from functools import partial

from utils.cell_phenotyping_utils import save_split

FOLDS_DIR = tutorials_dir / "example-data" / "cell-pheno-results" / "folds"
FOLDS_DIR.mkdir(parents=True, exist_ok=True)

# "2000" in the train filename records the budget the split was drawn under.
cap_tag = MAX_CELLS_PER_CLASS if MAX_CELLS_PER_CLASS is not None else "all"

write_split = partial(
    save_split,
    cell_ids=cells.index.values,
    cell_labels=cells["label"].values,
    out_dir=FOLDS_DIR,
)

for fold_id in (1, 2, 3, 4):
    train_idx, valid_idx, test_idx = split_fold(quadrant, fold_id, random_state=SEED)
    train_idx = cap_per_class(train_idx, y, MAX_CELLS_PER_CLASS, random_state=SEED)

    n_tr = write_split(train_idx, f"train_{cap_tag}_fold{fold_id}.csv")
    n_va = write_split(valid_idx, f"val_fold{fold_id}.csv")
    n_te = write_split(test_idx, f"test_fold{fold_id}.csv")
    print(f"fold {fold_id}: train {n_tr:>6,}  valid {n_va:>6,}  test {n_te:>6,}")

print("wrote ->", FOLDS_DIR)


##### Metrics

Four numbers, each answering a different question:

| Metric | Question it answers |
| --- | --- |
| **Macro F1** | Precision/recall balance, averaged over classes *equally* — the headline number under imbalance |
| **Balanced accuracy** | Mean per-class recall: is any class being ignored outright? |
| **Average precision** | Ranking quality of the predicted probabilities, threshold-free |
| **ROC AUC** (one-vs-rest) | Separability of each class from all others |

One subtlety worth spelling out: `predict_proba` returns columns in `clf.classes_` order, which omits any class absent from that fold's training rows. Scattering the probabilities into a fixed-width array keeps every column aligned to the same class across folds — otherwise the metrics silently compare different things.

In [ ]:
# evaluate — scores a fitted probe on the held-out cells: macro F1,
#            balanced accuracy, average precision, ROC AUC. Macro-averaging
#            weights every cell type equally, which is the point when the
#            rare classes are the interesting ones. `labels` pins the global
#            class order, because a fold may not train on all 16 classes.
from utils.cell_phenotyping_utils import evaluate


##### Run the probe over all four folds

Each fold: cap the training cells, tune `C`, refit, score on the held-out quadrant. Predictions are collected as we go, so at the end every cell on the slide has an **out-of-fold** prediction — one made by a probe that never saw that quadrant.

> **Runtime.** 15 trials × 4 folds = 60 logistic regressions per encoder, on up to 2000 × 16 = 32k cells × 768 dims. None of them are warm-started — TPE jumps around the search space, so there is no ascending path to exploit — and trials landing near the top of the range are barely regularized, ill-conditioned, and will grind toward `max_iter=10000`. Plan for the better part of an hour per encoder on a laptop CPU.
>
> To trim it: lower `MAX_CELLS_PER_CLASS`, lower `N_TRIALS` (but see the note above about dropping below 10), or lower `MAX_ITER`. All three depart from the published protocol — that is the trade.

Convergence warnings from the weakly-regularized trials are expected and are routed through `logging` above rather than silenced individually.

In [ ]:
# run_cross_validation — the whole protocol: for each quadrant, hold it
#                        out, cap the training budget, tune and fit the
#                        probe, then score it on the held-out cells. Every
#                        cell is tested exactly once, so the out-of-fold
#                        predictions cover the slide.
from utils.cell_phenotyping_utils import run_cross_validation

kronos_results, kronos_oof, kronos_trials = run_cross_validation(
    X_kronos,
    y,
    quadrant,
    name="KRONOS2",
    labels=LABELS,
    n_trials=N_TRIALS,
    c_range=C_RANGE,
    max_cells_per_class=MAX_CELLS_PER_CLASS,
    max_iter=MAX_ITER,
    seed=SEED,
    sampler_seed=SAMPLER_SEED,
)


#### 5. Results

Report **mean ± standard deviation across folds**, never a single fold. The spread is the informative part: it says how much the score depends on which region was held out, and a large spread means the headline mean is not something to quote with confidence.

In [ ]:
# summarize — appends Mean and Std Dev rows to a per-fold table. Drops C
#             first: averaging a tuned hyperparameter across folds
#             describes nothing, since each fold's C is only meaningful
#             against its own training set.
from utils.cell_phenotyping_utils import summarize

summarize(kronos_results)


##### Did the search actually find anything?

A hyperparameter search is only good if the objective varies over the searched range *and* the sampler concentrates where it is good. Both are checkable, and both are worth checking before trusting a tuned number — a flat curve means that the sampler never concentrated and brought you random draws.

Plotting validation F1 against `C` on a log axis answers the first. The trial *index* colouring answers the second: TPE's early trials are random by design, so later trials (darker) clustering near the optimum is the sampler working.

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
for ax, fold in zip(axes, kronos_trials["Fold"].unique()):
    t = kronos_trials[kronos_trials["Fold"] == fold]
    sc = ax.scatter(t["C"], t["valid_f1"], c=t.index, cmap="viridis", s=28)
    best = t.loc[t["valid_f1"].idxmax()]
    ax.axvline(best["C"], color="crimson", lw=0.8, ls="--")
    ax.set(xscale="log", xlabel="C", title=f"{fold}")
axes[0].set_ylabel("validation macro F1")
fig.colorbar(sc, ax=axes, label="trial index (later = darker)", fraction=0.02)
fig.suptitle("Optuna TPE search, per fold — dashed line = selected C")
plt.show()

#### 6. Baseline — is KRONOS2 earning its keep?

A probe score alone means nothing; the question is always *compared to what?* The natural baseline is the **mean-marker** readout Tutorial 3 also extracted: the average intensity of each marker over the cell's own pixels. That is the classic per-cell expression vector — no model, no GPU, and exactly what conventional cell phenotyping pipelines classify on.

Because Tutorial 3 extracted both encoders on the **same 18-marker panel**, this is a genuinely controlled comparison: same cells, same folds, same training budget, same search space, and the same input channels. KRONOS2 compresses those 18 markers into 768 learned dimensions; the baseline just averages them into 18. The only thing that differs is the representation.

In [ ]:
mean_marker = load_cell_features(slide, "mean_marker", cell_cfg)

# Re-apply the same row filters, in the same order, that X_kronos went through.
X_marker = np.asarray(mean_marker["features"].values, dtype=np.float32)
X_marker = X_marker[keep][on_grid]
assert X_marker.shape[0] == X_kronos.shape[0]
print("mean-marker features:", X_marker.shape, "(n_cells, 18 markers)")

marker_results, marker_oof, marker_trials = run_cross_validation(
    X_marker,
    y,
    quadrant,
    name="mean_marker",
    labels=LABELS,
    n_trials=N_TRIALS,
    c_range=C_RANGE,
    max_cells_per_class=MAX_CELLS_PER_CLASS,
    max_iter=MAX_ITER,
    seed=SEED,
    sampler_seed=SAMPLER_SEED,
    verbose=False,
)
summarize(marker_results)

In [ ]:
comparison = pd.DataFrame({
    "mean_marker": summarize(marker_results).loc["Mean"],
    "KRONOS2": summarize(kronos_results).loc["Mean"],
})
comparison["Δ"] = comparison["KRONOS2"] - comparison["mean_marker"]
comparison.round(4)

Read the delta, not the absolute numbers. Mean-marker features are a genuinely strong baseline on a well-designed antibody panel — the markers were *chosen* to separate these cell types — so a foundation model does not automatically win. Where KRONOS2 tends to pull ahead is on classes that intensity alone cannot resolve: types distinguished by subcellular localization or by morphology rather than by which channels are bright. The confusion matrix below is where you see which ones.

#### 7. Visual results

##### Confusion matrix

Row-normalized, so each row shows *where the cells of that true class actually went*. A strong diagonal is the goal; off-diagonal mass shows which pairs of phenotypes the representation cannot tell apart. Biologically adjacent confusions (M1 vs M2 macrophages, CD4 vs TReg) are the interesting ones.

In [ ]:
%matplotlib inline
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))
for ax, (name, oof) in zip(axes, [("mean_marker", marker_oof), ("KRONOS2", kronos_oof)]):
    cm = confusion_matrix(y, oof, labels=LABELS, normalize="true")
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set(
        title=f"{name} — out-of-fold confusion (row-normalized)",
        xlabel="predicted", ylabel="true",
        xticks=range(len(CLASS_NAMES)), yticks=range(len(CLASS_NAMES)),
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    )
    ax.tick_params(axis="x", rotation=90)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

##### Per-class F1

The macro average hides which classes carry it. Sorting per-class F1 by class frequency usually shows the expected story — rare phenotypes are the hard ones — and makes it obvious whether an encoder's advantage is broad or concentrated in a few types.

In [ ]:
%matplotlib inline

from sklearn.metrics import f1_score
 

per_class = pd.DataFrame({
    "KRONOS2": f1_score(y, kronos_oof, average=None, labels=LABELS, zero_division=0),
}, index=CLASS_NAMES)
per_class["n_cells"] = cells["label"].value_counts().reindex(CLASS_NAMES).values
per_class = per_class.sort_values("n_cells", ascending=False)

ax = per_class[["KRONOS2"]].plot.bar(figsize=(13, 4.5), width=0.8)
ax.set(ylabel="out-of-fold F1", xlabel="", title="Per-class F1 (classes ordered by abundance)")
# ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

per_class.round(3)

##### Predictions in space

Every cell carries a centroid, so the out-of-fold predictions paint straight back onto the slide. Comparing the ground-truth map against the predicted map is the check no aggregate metric gives you: **are the errors scattered, or do they cluster?** Scattered errors are ordinary classifier noise. A whole region predicted wrong points at something structural — a staining gradient, a tissue compartment unlike anything in the training quadrants, or a fold whose held-out region was simply too different.

In [ ]:
%matplotlib inline
from matplotlib.colors import ListedColormap

cmap = ListedColormap(plt.get_cmap("tab20").colors[:len(CLASS_NAMES)])
correct = kronos_oof == y

fig, ax = plt.subplots(1, 3, figsize=(21, 7))
for a, values, title in [
    (ax[0], y, "Ground truth"),
    (ax[1], kronos_oof, "KRONOS2 out-of-fold prediction"),
]:
    a.scatter(cells["x"], cells["y"], c=values, cmap=cmap, s=0.4,
              vmin=-0.5, vmax=len(CLASS_NAMES) - 0.5)
    a.set_title(title)

ax[2].scatter(cells["x"][correct], cells["y"][correct], c="lightgrey", s=0.4)
ax[2].scatter(cells["x"][~correct], cells["y"][~correct], c="crimson", s=0.6)
ax[2].set_title(f"Errors in red ({(~correct).mean():.1%} of cells)")

for a in ax:
    a.invert_yaxis()
    a.set_aspect("equal")
    a.axis("off")

handles = [plt.Line2D([], [], marker="o", ls="", color=cmap(i), label=n)
           for i, n in enumerate(CLASS_NAMES)]
fig.legend(handles=handles, loc="lower center", ncol=8, frameon=False)
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

#### 8. Save the results

Three results worth keeping: the full Optuna trial history per encoder, the per-fold metric tables, and the `C` each fold's search selected.


In [ ]:
RESULTS_DIR = tutorials_dir / "example-data" / "cell-pheno-results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

kronos_trials.to_csv(RESULTS_DIR / "optuna_results_kronos2.csv", index=False)
marker_trials.to_csv(RESULTS_DIR / "optuna_results_mean_marker.csv", index=False)

# The C each fold's search settled on. Tutorial 6 inherits the KRONOS2
# column so its frozen baseline matches this probe rather than re-searching.
best_c = pd.DataFrame({
    "KRONOS2": kronos_results["C"],
    "mean_marker": marker_results["C"],
})
best_c.to_csv(RESULTS_DIR / "best_c.csv")

average_results = pd.concat(
    {"KRONOS2": summarize(kronos_results), "mean_marker": summarize(marker_results)},
    names=["encoder", "Fold"],
)
average_results.to_csv(RESULTS_DIR / "average_results.csv")
print("wrote ->", RESULTS_DIR)
print("\nselected C per fold:")
print(best_c.to_string(float_format=lambda v: f"{v:.3e}"))
average_results


##### Optional — write the predictions back to the slide

Predictions use the **same schema** as ground-truth labels (`cell_id`, `label_set`, `label`), written to their own file so the annotations stay untouched. That makes them readable by the same tooling — including `visualize_cells_inset`, which will render a zoomed view tinted by *predicted* phenotype for direct comparison against Tutorial 3's ground-truth inset of the same box.

In [ ]:
predictions = pd.DataFrame({
    "cell_id": cells.index.values,
    "label_set": "kronos2_probe",
    "label": np.asarray(CLASS_NAMES)[kronos_oof],
})
pred_csv = slide.path / "cells" / "cell_predictions.csv"
predictions.to_csv(pred_csv, index=False)
print(f"wrote {len(predictions):,} predictions -> {pred_csv}")

from IPython.display import display
from PIL import Image

_, h, w = slide.image.shape
inset = slide.visualize_cells_inset(
    top_left=(w // 2, h // 2),   # (x, y) level-0 pixels — same box as Tutorial 3
    box_size=(400, 400),
    labels_csv=pred_csv,
)
display(Image.open(inset))

#### 9. Recap

Starting from features Tutorial 3 had already computed, you built spatially honest folds, fit a regularization-tuned linear probe on frozen KRONOS2 embeddings, scored it against a mean-marker baseline on identical splits, and mapped the errors back onto the tissue.

The result is a **benchmark number with a defensible protocol behind it** — geographic folds with a guard band, an equal training budget per encoder, hyperparameters chosen on validation and never on test, and mean ± std across folds rather than a lucky single split. Those choices are what make the comparison mean something; the logistic regression itself is the easy part.

The wider lesson is that "the same benchmark" is a claim about half a dozen independent choices — which classes are scored, how many cells the probe sees, how folds are drawn, what `C` range is searched, how many trials, which encoder. A reproduction can fail on any one of them while looking perfectly reasonable, and the quiet ones (the class list, especially) do more damage than the loud ones.

**Where next?**

- **Swap the encoder.** Any CORAL extractor writes the same `(n_cells, d)` matrix with the same `cell_id` coordinate, so re-running `run_cross_validation` on a different encoder's features is a one-line change. The protocol above is the harness.
- **Vary the training budget.** Re-run with `MAX_CELLS_PER_CLASS` at 100, 200, 500, 1000, 2000 to trace a label-efficiency curve. A representation that holds up at 100 cells per class is worth far more in practice than one that only wins with the full set — this is where foundation models usually separate from hand-crafted features.
- **Check the seed sensitivity.** Set `SEED_SAMPLER = False` and run three times. The spread tells you how much of any single decimal is signal.
- **Vary the patch size.** 64 px encloses one cell; a larger box lets the encoder see the neighbourhood too. Whether that helps or hurts is an empirical question this notebook can answer directly.
- **Move to cohorts.** With several slides, fold on **slide** (or patient) instead of quadrant — the same code, with `quadrant` replaced by a slide id. That measures generalization across batches and patients, which is the number that actually predicts clinical utility.
- **Beyond linear.** A linear probe is deliberately weak, to isolate representation quality. If you want maximum accuracy rather than a clean measurement, a small MLP head or full fine-tuning on the same folds is the next rung.